Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Interactive charts: widgets and selections

So far, our charts have had one kind of interaction: the **tooltip**. This notebook adds the rest, in Altair. It follows the lecture practical (the Vega-Lite files in `weeks/week09/examples/lecture`) step by step, then adds the lecture's other interactive examples.

There are two kinds of interaction:

* **Widgets:** the reader sets a value with a control under the chart: a dropdown, radio buttons, a slider, a checkbox, a colour picker.
* **Selections:** the reader acts on the chart itself: hovering, clicking the legend, dragging across it.

Both work the same way. **Every interaction is a param plus a place that uses it.** A param is a named value that the chart can change while someone is looking at it. You use it in a filter, a calculation, a mark property or a condition, and the chart updates when it changes.

In this notebook:
1. **Widgets:** dropdown, radio buttons, range slider, checkbox, colour picker
2. **Selections:** hover highlight, clickable legend, brush to zoom
3. **Putting them together:** a slider and a legend; a dropdown map; a re-indexing slider
4. **Saving** an interactive chart for your website

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import pandas as pd
import altair as alt

DATA = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week09/data/"
MENU = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week08/data/"     # Week 8's menu prices
MAPS = "https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/gbr/"

SOURCE = "Source: LSE Growth Lab, AutoCPI (menu photos)"     # for chart subtitles
BROWN, BAND = "#8c5a2b", "#ecdfd0"     # colours for the menu charts


<br>

The main dataset is from Week 8: the price of 12 dishes on UK menus, summarised by year (the median, and the 10th, 25th, 75th and 90th percentiles). It's small, so we read it with pandas and add a date for the time axis:

In [ ]:
prices = pd.read_csv(MENU + "menu_percentiles_by_year.csv")
prices["date"] = pd.to_datetime(prices["year"], format="%Y")     # 1 January of each year, for a time axis

dishes = sorted(prices["item"].unique())
print(dishes)
prices.head()

<br><br>

---

<br>

## 1. Widgets

### 1.1 A dropdown

Start with a chart you know: the median price of a latte. The dish is **hard-coded** in the filter:

In [ ]:
alt.Chart(prices).mark_line(color=BROWN, point=alt.OverlayMarkDef(color=BROWN)).transform_filter(
    alt.datum.item == "latte"
).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("median:Q").title("£").scale(zero=False),
).properties(width=450, height=250, title="The price of a latte")

<br>

**Step 1: name the value.** Make a param called `dish`, starting at `"latte"`, and use it where `"latte"` was. `add_params` attaches it to the chart. Run this: **nothing changes**. The chart is the same, but the value now has a name the chart can change.

In [ ]:
dish = alt.param(name="dish", value="latte")

alt.Chart(prices).mark_line(color=BROWN, point=alt.OverlayMarkDef(color=BROWN)).transform_filter(
    alt.datum.item == dish                     # the param, where the hard-coded value was
).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("median:Q").title("£").scale(zero=False),
).add_params(
    dish
).properties(width=450, height=250, title="The price of a latte")

<br>

**Step 2: bind it to a widget.** `bind` gives the param a control. `binding_select` makes a dropdown with the options you list. The title can use the param too, with an **expression** (`alt.expr`): text in single quotes, joined with `+`:

In [ ]:
dish = alt.param(
    name="dish",
    value="latte",
    bind=alt.binding_select(options=dishes, name="Dish "),
)

alt.Chart(prices).mark_line(color=BROWN, point=alt.OverlayMarkDef(color=BROWN)).transform_filter(
    alt.datum.item == dish
).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("median:Q").title("£").scale(zero=False),
    tooltip=[
        alt.Tooltip("year:O").title("Year"),
        alt.Tooltip("median:Q").format(".2f").title("Median (£)"),
    ],
).add_params(
    dish
).properties(
    width=450, height=250,
    title=alt.Title(alt.expr("'The price of a ' + dish"), subtitle=SOURCE),
)

<br>

> **Don't call a param `item`.** Vega uses that name for itself, and the chart breaks with *"item is not defined"*. That's why ours is called `dish`.

### 1.2 Radio buttons

A param can hold a **column name**. Here the reader picks which percentile to plot. `datum[measure]` means "the column whose name is in `measure`". `labels` gives each option a friendlier name:

In [ ]:
measure = alt.param(
    name="measure",
    value="median",
    bind=alt.binding_radio(
        options=["p10", "p25", "median", "p75", "p90"],
        labels=["10th", "25th", "Median", "75th", "90th"],
        name="Percentile ",
    ),
)

alt.Chart(prices).mark_line(color=BROWN, point=alt.OverlayMarkDef(color=BROWN)).transform_filter(
    alt.datum.item == "latte"
).transform_calculate(
    value="datum[measure]"                      # the column the reader picked
).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("value:Q").title("£").scale(domain=[0, 6]),
).add_params(
    measure
).properties(width=450, height=250,
             title=alt.Title("The price of a latte: cheap, typical and dear", subtitle=SOURCE))

<br>

The y-axis is fixed (`domain=[0, 6]`), so the line moves up and down as you switch. Without it, the axis would rescale and the line would look the same each time.

### 1.3 A range slider

`binding_range` makes a slider: give it a `min`, `max` and `step`. Use the param in a filter to choose where the chart starts:

In [ ]:
start = alt.param(
    name="startYear",
    value=2017,
    bind=alt.binding_range(min=2017, max=2024, step=1, name="From "),
)

alt.Chart(prices).mark_line(color=BROWN, point=alt.OverlayMarkDef(color=BROWN)).transform_filter(
    (alt.datum.item == "latte") & (alt.datum.year >= start)
).encode(
    x=alt.X("date:T").title(None).axis(format="%Y", tickCount="year"),
    y=alt.Y("median:Q").title("£").scale(zero=False),
).add_params(
    start
).properties(width=450, height=250,
             title=alt.Title(alt.expr("'The price of a latte since ' + startYear"), subtitle=SOURCE))

<br>

`tickCount="year"` puts at most one tick on each year. Without it, a short time axis can show the same year twice.

Two sliders make a window. Here, the price of Bitcoin, with a start and an end year:

In [ ]:
bitcoin = alt.Data(url=DATA + "bitcoin_price.csv", format=alt.DataFormat(parse={"Date": "date", "Price": "number"}))

min_year = alt.param(name="minYear", value=2020, bind=alt.binding_range(min=2010, max=2024, step=1, name="From "))
max_year = alt.param(name="maxYear", value=2024, bind=alt.binding_range(min=2010, max=2024, step=1, name="To "))

alt.Chart(bitcoin).mark_line(color="#f2a900", strokeWidth=1.4).transform_filter(
    (alt.expr.year(alt.datum.Date) >= min_year) & (alt.expr.year(alt.datum.Date) <= max_year)
).encode(
    x=alt.X("Date:T").title(None),
    y=alt.Y("Price:Q").title(None).axis(format="$,.0f"),
).add_params(
    min_year, max_year
).properties(width=450, height=250,
             title=alt.Title("The price of Bitcoin", subtitle="Daily closing price, US$. Source: Investing.com"))

<br>

### 1.4 A checkbox

A checkbox gives `true` or `false`. Use it in an expression with `? :` ("if ... then ... else ..."). Here, the band for the middle 80% of prices appears when the box is ticked: its opacity is 1 if `showRange` is true, and 0 if not:

In [ ]:
show_range = alt.param(
    name="showRange",
    value=False,
    bind=alt.binding_checkbox(name="Show the middle 80% of prices "),
)

latte = alt.Chart(prices).transform_filter(alt.datum.item == "latte")

band = latte.mark_area(color=BAND, opacity=alt.expr("showRange ? 1 : 0")).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("p10:Q").title("£"),
    y2="p90:Q",
)
line = latte.mark_line(color=BROWN, point=alt.OverlayMarkDef(color=BROWN)).encode(
    x="date:T",
    y="median:Q",
)

(band + line).add_params(show_range).properties(
    width=450, height=250, title=alt.Title("The price of a latte", subtitle=SOURCE))

<br>

A checkbox can change almost anything. A classic use: whether the y-axis starts at zero. Scale settings like `zero` take an expression too:

In [ ]:
gdp = alt.Data(url=DATA + "us_real_gdp_per_capita.csv", format=alt.DataFormat(parse={"Date": "date", "Value": "number"}))

from_zero = alt.param(name="fromZero", value=False, bind=alt.binding_checkbox(name="Start the y-axis at 0 "))

alt.Chart(gdp).mark_line(color="#800000").transform_filter(
    "year(datum.Date) >= 2000"
).encode(
    x=alt.X("Date:T").title(None),
    y=alt.Y("Value:Q").title(None).axis(format="$,.0f").scale(zero=alt.expr("fromZero")),
).add_params(
    from_zero
).properties(width=450, height=250,
             title=alt.Title("US real GDP per person", subtitle="US$, 2017 prices. Source: FRED"))

<br>

Tick and untick it. The same data, two very different impressions: this is why the choice matters.

### 1.5 A colour picker

`alt.binding(input="color")` gives the browser's colour picker. A colour input only accepts a **hex code**, so the starting value is `"#c8102e"`, not `"red"`. The param sets the mark's colour directly:

In [ ]:
inflation = alt.Data(url=DATA + "ons_rpi_inflation_long_run.json",
                     format=alt.DataFormat(type="json", property="years", parse={"date": "date", "value": "number"}))

colour = alt.param(name="colour", value="#c8102e", bind=alt.binding(input="color", name="Colour "))

alt.Chart(inflation).mark_line(color=alt.expr("colour"), strokeWidth=1.5).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("value:Q").title("%"),
).add_params(
    colour
).properties(width=450, height=250,
             title=alt.Title("Inflation since 1800", subtitle="Retail prices index, annual % change. Source: ONS (CDSI)"))

<br>

`property="years"` picks the annual rows from the ONS file, which also has monthly and quarterly ones.

<br><br>

---

<br>

## 2. Selections

A selection is a param that the reader sets by acting on the chart. The recipe has three steps:

1. **Make a selection:** `alt.selection_point` (pick things) or `alt.selection_interval` (drag across a range).
2. **Use it in a condition:** `alt.when(selection).then(...).otherwise(...)`.
3. **Choose the trigger:** a click (the default), `on="pointerover"`, or the legend.

For these, we compare dishes, so first **rebase** each dish to 2019 = 100 (more on this in DS9_2):

In [ ]:
base = prices[prices["year"] == 2019].set_index("item")["median"]     # each dish's 2019 price
prices["index"] = prices["median"] / prices["item"].map(base) * 100
prices[["item", "year", "median", "index"]].head()

<br>

### 2.1 Hover highlight

Hover over a line to pick it out. The selection picks a **dish** (`fields=["item"]`), and `nearest=True` picks the closest point, so you don't have to land exactly on a line. A selection needs points to choose from, so it goes on a layer of (invisible) points:

In [ ]:
hover = alt.selection_point(fields=["item"], on="pointerover", nearest=True)

base = alt.Chart(prices).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("index:Q").title("Index, 2019 = 100").scale(zero=False),
    color=alt.Color("item:N").title(None).scale(scheme="tableau20"),
)

lines = base.mark_line().encode(
    opacity=alt.when(hover).then(alt.value(1)).otherwise(alt.value(0.15)),
)
points = base.mark_point(opacity=0).encode(
    tooltip=[
        alt.Tooltip("item:N").title("Dish"),
        alt.Tooltip("year:O").title("Year"),
        alt.Tooltip("index:Q").format(".0f").title("Index (2019 = 100)"),
    ],
).add_params(hover)

(lines + points).properties(width=450, height=300,
                            title=alt.Title("Menu prices since 2019", subtitle=["Hover to pick out a dish. " + SOURCE]))

<br>

### 2.2 A clickable legend

One change from 2.1: `bind="legend"`. Now clicking a dish in the legend selects it (shift-click for more than one). No points layer is needed:

In [ ]:
pick = alt.selection_point(fields=["item"], bind="legend")

alt.Chart(prices).mark_line(point=True).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("index:Q").title("Index, 2019 = 100").scale(zero=False),
    color=alt.Color("item:N").title(None).scale(scheme="tableau20"),
    opacity=alt.when(pick).then(alt.value(1)).otherwise(alt.value(0.1)),
).add_params(
    pick
).properties(width=450, height=300,
             title=alt.Title("Menu prices since 2019", subtitle=["Click a dish in the legend. " + SOURCE]))

<br>

### 2.3 Brush to zoom

An **interval** selection lets the reader drag across a chart. `encodings=["x"]` limits it to the x direction. Here the small chart below is the control: drag across it, and the big chart above shows only that period, using the selection as a **filter**. The two charts are stacked with `&` (vertical concatenation):

In [ ]:
long_run = alt.Data(url=DATA + "uk_prices_long_run.csv")
brush = alt.selection_interval(encodings=["x"])

base = alt.Chart(long_run).transform_calculate(date="datetime(datum.year, 0, 1)")

detail = base.mark_line(color="#1f4e79").encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("index:Q").title("1974 = 100").scale(type="log"),
    tooltip=[
        alt.Tooltip("year:O").title("Year"),
        alt.Tooltip("index:Q").format(",.1f").title("Index (1974 = 100)"),
    ],
).transform_filter(brush).properties(width=500, height=250)

overview = base.mark_line(color="#999").encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("index:Q").title(None).scale(type="log").axis(labels=False, ticks=False, grid=False),
).add_params(brush).properties(width=500, height=60)

(detail & overview).properties(
    title=alt.Title("UK prices since 1800", subtitle=["Retail prices index (log scale). Drag across the lower chart to zoom. Source: ONS"]))

<br>

The y-axis is a **log scale**: equal steps are equal *percentage* changes, so 1800s inflation and 1970s inflation can be compared on one chart.

<br><br>

---

<br>

## 3. Putting them together

### 3.1 A slider and a legend

Maternal mortality and GDP per person, one year at a time. The slider picks the year (a filter); the legend picks the regions (a condition on opacity). `order` draws the selected points on top:

In [ ]:
mortality = alt.Data(url=DATA + "maternal_mortality_gdp.json")

year = alt.param(name="year", value=2010, bind=alt.binding_range(min=2000, max=2017, step=1, name="Year "))
region = alt.selection_point(fields=["Region"], bind="legend")

alt.Chart(mortality).mark_circle().transform_filter(
    alt.datum.Year == year
).encode(
    x=alt.X("GDP pc:Q").title("GDP per person (PPP, log scale)").scale(type="log", domain=[400, 150000]).axis(format="$,"),
    y=alt.Y("Maternal Mortality:Q").title("Maternal deaths per 100,000 live births (log scale)").scale(type="log"),
    color=alt.Color("Region:N").title(None).legend(orient="top", columns=3),
    size=alt.Size("Population:Q").scale(range=[12, 1200]).legend(None),
    opacity=alt.when(region).then(alt.value(0.9)).otherwise(alt.value(0.1)),
    order=alt.when(region).then(alt.value(1)).otherwise(alt.value(0)),
    tooltip=[
        alt.Tooltip("Country:N").title("Country"),
        alt.Tooltip("GDP pc:Q").format("$,.0f").title("GDP per person"),
        alt.Tooltip("Maternal Mortality:Q").format(",").title("Maternal mortality"),
    ],
).add_params(
    year, region
).properties(width=450, height=320,
             title=alt.Title("GDP and maternal mortality", subtitle="Source: WHO, World Bank"))

<br>

### 3.2 A dropdown map

A param can pick the column to **map**. The ITL3 areas come from map-data (as in Week 7); `transform_lookup` joins the indicators by area code, then `datum[series]` picks the one in the dropdown. A **quantile** colour scale puts a quarter of the areas in each colour, so one extreme area (the City of London) doesn't wash out the rest. Its legend doesn't draw well, so we leave it out: darker is higher, and the tooltip gives the value:

In [ ]:
itl3 = alt.Data(url=MAPS + "ITL3_UK_2021.json", format=alt.DataFormat(type="topojson", feature="geog"))
indicators = alt.Data(url=DATA + "itl3_indicators.csv")

options = ["GDP per capita (£)", "GDP Growth 2010-19 (%)", "Employment (%)", "Unemployment (%)", "Economic Inactivity (%)"]
series = alt.param(name="series", value=options[0], bind=alt.binding_select(options=options, name="Indicator "))

alt.Chart(itl3).mark_geoshape(stroke="white", strokeWidth=0.3).transform_lookup(
    lookup="properties.id",
    from_=alt.LookupData(data=indicators, key="ITL code", fields=["Area"] + options),
).transform_calculate(
    value="datum[series]"
).encode(
    color=alt.Color("value:Q").scale(type="quantile", scheme="goldred").legend(None),
    tooltip=[
        alt.Tooltip("Area:N").title("Area"),
        alt.Tooltip("value:Q").format(",.1f").title("Value"),
    ],
).add_params(
    series
).project("mercator").properties(width=320, height=420,
                                 title=alt.Title(alt.expr("series"), subtitle="ITL3 areas. Source: ONS"))

<br>

### 3.3 A re-indexing slider

The lecture's fig 7: pick a year, and every value is divided by that year's value. A `window` transform with `first_value` finds the value in the first year that's left after the filter:

In [ ]:
index_year = alt.param(name="indexYear", value=1974, bind=alt.binding_range(min=1800, max=2024, step=1, name="Index year "))

alt.Chart(long_run).transform_filter(
    alt.datum.year >= index_year
).transform_window(
    first="first_value(index)",
    sort=[alt.SortField("year")],
    frame=[None, None],
).transform_calculate(
    rebased="datum.index / datum.first * 100",
    date="datetime(datum.year, 0, 1)",
).mark_line(color="#179fdb").encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("rebased:Q").title(None),
    tooltip=[
        alt.Tooltip("year:O").title("Year"),
        alt.Tooltip("rebased:Q").format(",.0f").title("Index"),
    ],
).add_params(
    index_year
).properties(width=450, height=250,
             title=alt.Title(alt.expr("'UK prices, ' + indexYear + ' = 100'"), subtitle="Retail prices index. Source: ONS"))

<br><br>

---

<br>

## 4. Saving an interactive chart

Interactive charts save like any other: `chart.save("name.json")`, then embed the JSON on your website with `vegaEmbed`, as in Week 1. The widgets come with the chart. Save the dropdown chart from section 1.1:

In [ ]:
chart = alt.Chart(prices).mark_line(color=BROWN, point=alt.OverlayMarkDef(color=BROWN)).transform_filter(
    alt.datum.item == dish
).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("median:Q").title("£").scale(zero=False),
).add_params(
    dish
).properties(width=450, height=250, title=alt.Title(alt.expr("'The price of a ' + dish"), subtitle=SOURCE))

chart.save("menu_prices_dropdown.json")

<br>

> **Data and interactive charts.** A pandas DataFrame is saved **inside** the JSON. That's fine for a small summary like `prices` (120 rows), but for big data, summarise first (Week 8), or point the chart at a URL so the data stays in its own file.

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1. Two widgets.** Add the dish dropdown (1.1) to the checkbox chart (1.4), so the reader picks a dish *and* whether to show its range.

**2. A rebasing slider.** In section 2, the base year is fixed at 2019. Make it a slider (2017 to 2026), and use `joinaggregate` to find each dish's price in that year: see `weeks/week09/examples/lecture/L4_rebase_slider.json`.

**3. Hover on a scatter.** Make a scatter of GDP per person against life expectancy (`gdp_life_expectancy.csv`), and highlight a continent when you hover over one of its countries.

**4. Your CC9 chart.** Take a chart from your portfolio and add one widget and one selection. What does each let the reader find out that a static chart doesn't?

In [ ]:
# Your code here
